# Lezione 4 — Pratica: plateau sterili

In questa pratica verificherai in prima persona come la scelta dell'operatore di lettura influenzi la gravità del plateau sterile (barren plateau).

## Traccia

Nell'esperimento della lezione, l'operatore di lettura è la somma di un operatore Z per ciascun qubit: `sum(cirq.Z(q) for q in qubits)`. Il tuo compito è sostituirlo con un singolo operatore che combini due qubit: `cirq.Z(qubits[0]) * cirq.Z(qubits[1])`. Poi riesegui la scansione delle dimensioni dei qubit e confronta la nuova curva della varianza con quella originale.

## Suggerimenti

- L'operatore di lettura si passa come lista al parametro `operators` di `tfq.layers.SampledExpectation`: con la somma di Z devi racchiuderla tra parentesi quadre: `[sum(cirq.Z(q) for q in qubits)]`
- L'operatore Z⊗Z è già naturalmente racchiuso in una lista con un solo elemento: `[cirq.Z(qubits[0]) * cirq.Z(qubits[1])]`
- Osserva se la nuova curva scende più rapidamente, più lentamente o allo stesso modo dell'originale, e collega ciò che osservi alla raccomandazione di "scegliere con cura cosa misurare" vista nella teoria

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt

def generar_circuito_aleatorio(qubits, profundidad, parametro):
    """Un circuito con una puerta aleatoria distinta en cada qubit y en cada capa,
    entrelazado con puertas CZ. Solo el primer parámetro es simbólico (entrenable);
    el resto son ángulos aleatorios fijos."""
    circuito = cirq.Circuit()
    for qubit in qubits:
        circuito += cirq.ry(np.pi / 4.0)(qubit)

    for capa in range(profundidad):
        for i, qubit in enumerate(qubits):
            es_el_parametro_entrenable = (i == 0 and capa == 0)
            angulo = parametro if es_el_parametro_entrenable else np.random.uniform(0, 2 * np.pi)
            eleccion = np.random.uniform()
            if eleccion > 2 / 3:
                circuito += cirq.rz(angulo)(qubit)
            elif eleccion > 1 / 3:
                circuito += cirq.ry(angulo)(qubit)
            else:
                circuito += cirq.rx(angulo)(qubit)
        for origen, destino in zip(qubits, qubits[1:]):
            circuito += cirq.CZ(origen, destino)
    return circuito

def varianza_del_gradiente(num_qubits, profundidad, num_circuitos, repeticiones, operador_lectura):
    """Igual que en la clase, pero ahora recibe una función `operador_lectura(qubits)`
    que construye el operador de lectura, para poder comparar distintas elecciones."""
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    simbolo = sympy.symbols("parametro")
    circuitos = [generar_circuito_aleatorio(qubits, profundidad, simbolo) for _ in range(num_circuitos)]

    operadores = operador_lectura(qubits)
    diferenciador = tfq.differentiators.ParameterShift()
    capa_expectativa = tfq.layers.SampledExpectation(differentiator=diferenciador)

    circuitos_tensor = tfq.convert_to_tensor(circuitos)
    valores = tf.convert_to_tensor(np.random.uniform(0, 2 * np.pi, (num_circuitos, 1)).astype(np.float32))

    with tf.GradientTape() as cinta:
        cinta.watch(valores)
        salida = capa_expectativa(circuitos_tensor, operators=operadores,
                                   repetitions=repeticiones, symbol_names=[simbolo], symbol_values=valores)

    gradientes = cinta.gradient(salida, valores)
    return tf.math.reduce_std(gradientes, axis=0).numpy()[0]

# Parámetros del barrido (igual que en la clase)
num_circuitos = 20
repeticiones = 500
tamanos_qubits = [2, 4, 6, 8]
profundidad_fija = 20

# Operador original de la clase: suma de un Z por cada qubit
def operador_suma_z(qubits):
    return [sum(cirq.Z(q) for q in qubits)]

varianzas_suma_z = [varianza_del_gradiente(n, profundidad_fija, num_circuitos, repeticiones, operador_suma_z) for n in tamanos_qubits]

In [ ]:
# TODO: define una función `operador_zz(qubits)` que devuelva el operador de lectura
# cirq.Z(qubits[0]) * cirq.Z(qubits[1]) envuelto en una lista de un elemento

# TODO: vuelve a recorrer `tamanos_qubits` llamando a varianza_del_gradiente con tu nuevo
# operador y guarda el resultado en una lista `varianzas_zz`

# TODO: dibuja ambas curvas (varianzas_suma_z y varianzas_zz) en la misma gráfica,
# con etiquetas y leyenda, para comparar la caída de varianza

<details>
<summary>💡 Mostra soluzione</summary>

```python
# Nuovo operatore di lettura: prodotto Z0*Z1 invece della somma di Z
def operador_zz(qubits):
    return [cirq.Z(qubits[0]) * cirq.Z(qubits[1])]

# Ripetiamo la scansione dei qubit con il nuovo operatore
varianzas_zz = [varianza_del_gradiente(n, profundidad_fija, num_circuitos, repeticiones, operador_zz) for n in tamanos_qubits]

# Confrontiamo le due curve
plt.plot(tamanos_qubits, varianzas_suma_z, marker="o", label="somma di Z, uno Z per qubit")
plt.plot(tamanos_qubits, varianzas_zz, marker="o", label="Z0 * Z1")
plt.xlabel("Numero di qubit")
plt.ylabel("Varianza del gradiente")
plt.title("Confronto dell'operatore di lettura sul plateau sterile")
plt.legend()
plt.show()

# La curva di Z0 * Z1 scende più rapidamente di quella di sum(Z): misurare un operatore
# che combina più qubit contemporaneamente soffre di un plateau sterile più severo
# rispetto alla misura della somma di operatori locali, come anticipava la teoria.
```

</details>